# Lab 7 · Serving B — vLLM

**~25 minutes.**

> ### ⚠️ Read this first
>
> **Start this notebook in a fresh session** (Run → Restart & Clear Cell
> Outputs, or open it as a new notebook). vLLM pins its own torch build
> and will fight the Unsloth environment if both are installed.
>
> vLLM is also tight on a 15 GB T4. If it won't start, that's a resource
> limit, not a mistake on your part — read along, and use
> `docs/troubleshooting.md`. Lab 6 already gave you a working served
> model, so nothing here is load-bearing.

Ollama is for running a model. vLLM is for *serving* one — many
concurrent users, maximum throughput.

> ↳ Slides: *Parameters / KV Cache* · *Inference Parameters*

## 8.1 Why a serving engine exists

In Lab 1 we watched the KV cache trade memory for speed. Serving many
users at once turns that into the central engineering problem.

The naive approach reserves a contiguous block of KV cache per request,
sized for the longest output it *might* produce. Most requests finish
early, so most of that memory is never used — reported waste is often
60–80%.

**PagedAttention** borrows from operating systems: split the cache into
fixed-size blocks and keep a block table per sequence, so the cache can be
non-contiguous and allocated on demand. Waste drops to a few percent, and
the memory you get back becomes concurrency.

On top of that, **continuous batching** admits a new request the moment
any sequence finishes, instead of waiting for the whole batch. That is
what section 8.6 measures.

## Install vLLM

3–5 minutes. It will install its own torch — expected here, and the
reason this notebook needs its own session.

In [ ]:
!pip install -q vllm openai
print("\ninstalled")

In [ ]:
# --- Locate the workshop repo -------------------------------------------
# Tries, in order: already present -> attached Kaggle Dataset -> git clone.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ankush610/LLM-lab.git"

def find_repo() -> Path:
    for candidate in [Path("/kaggle/working/LLM-lab"), Path.cwd(), Path.cwd().parent]:
        if (candidate / "common" / "config.py").exists():
            return candidate
    for d in Path("/kaggle/input").glob("*"):          # attached as a Dataset
        if (d / "common" / "config.py").exists():
            return d
    print("Repo not found locally, cloning...")        # last resort
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/LLM-lab"], check=True)
    return Path("/kaggle/working/LLM-lab")

REPO = find_repo()
sys.path[:0] = [str(REPO / "common"), str(REPO / "dataset")]
print(f"repo: {REPO}")

import config
print(config.summary())

In [ ]:
# --- Hugging Face authentication ----------------------------------------
# Reads the Kaggle Secret named HF_TOKEN. Never paste a token into a cell:
# it is saved with the notebook and shared notebooks leak tokens constantly.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face: authenticated")
except Exception as e:
    print(f"Hugging Face: NOT authenticated ({type(e).__name__})")
    print("  Fix: right panel -> Add-ons -> Secrets -> add HF_TOKEN, tick the box.")
    print("  Or set USE_UNGATED_MODEL = True below to skip the gated model entirely.")

## 8.2 Serving the adapter without merging

vLLM can load a base model once and apply LoRA adapters per request
(`--enable-lora`). With forty fine-tuned variants that's one 6 GB model in
memory and forty 90 MB adapters, switchable per request — rather than
forty full models.

We point it at the **base** model plus our adapter directory.

## 8.3 The flags that matter

| flag | why |
|---|---|
| `--dtype half` | **mandatory on T4** — sm75 has no bf16 |
| `--max-model-len 2048` | KV cache scales with this; the default would OOM |
| `--gpu-memory-utilization 0.85` | fraction of VRAM vLLM may claim |
| `--enable-lora` | turn on adapter support |
| `--max-lora-rank 16` | must be ≥ the `r` we trained with |

In [ ]:
import subprocess, time, requests, os

BASE_MODEL = config.MODEL_NAME.replace("unsloth/", "").replace("-bnb-4bit", "")
BASE_MODEL = f"meta-llama/{BASE_MODEL}" if "Llama" in BASE_MODEL else f"Qwen/{BASE_MODEL}"
print(f"base model : {BASE_MODEL}")
print(f"adapter    : {config.ADAPTER_DIR}")

cmd = [
    "vllm", "serve", BASE_MODEL,
    "--dtype", "half",                       # T4: no bf16
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.85",
    "--enable-lora",
    "--lora-modules", f"aura={config.ADAPTER_DIR}",
    "--max-lora-rank", str(config.LORA_R),
    "--port", str(config.VLLM_PORT),
    "--disable-log-requests",
]
print("\n" + " ".join(cmd))

log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

### Wait for the server

Loading and profiling the KV cache takes **3–6 minutes**. Poll `/health`
rather than guessing at a sleep.

In [ ]:
url = f"http://127.0.0.1:{config.VLLM_PORT}"
ready = False

for attempt in range(90):                     # up to ~7.5 minutes
    if server.poll() is not None:             # process died
        print("server exited - last 30 log lines:")
        !tail -30 /kaggle/working/vllm.log
        break
    try:
        if requests.get(f"{url}/health", timeout=2).status_code == 200:
            print(f"\nvLLM ready after ~{attempt*5}s")
            ready = True
            break
    except requests.exceptions.RequestException:
        pass
    if attempt % 6 == 0:
        print(f"  waiting... {attempt*5}s")
    time.sleep(5)

if not ready and server.poll() is None:
    print("timed out; last 30 log lines:")
    !tail -30 /kaggle/working/vllm.log

## 8.5 The OpenAI client

vLLM speaks the OpenAI API. Third and final pass over the sampling
parameters — and note that `presence_penalty` and `frequency_penalty` are
**separate fields** here, unlike HuggingFace's single
`repetition_penalty`.

`model="aura"` selects our LoRA adapter by the name we registered.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"{url}/v1", api_key="not-needed")

def chat(prompt, model="aura", **params):
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": config.SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        max_tokens=params.pop("max_tokens", 256),
        **params,
    )
    return r.choices[0].message.content

from eval_prompts import EVAL_PROMPTS
for p in EVAL_PROMPTS[:3]:
    print(f"Q: {p['question']}")
    print(f"A: {chat(p['question'], temperature=0).strip()[:300]}\n")

### Adapter on, adapter off — one server

Same loaded weights. `model="aura"` applies the LoRA; using the base model
name skips it. This is the deployment story from Lab 4's toggle, running
as a service.

In [ ]:
q = "Where are the shared Apptainer images stored on AURA?"
print("--- with adapter (model='aura') ---")
print(chat(q, model="aura", temperature=0).strip()[:280])
print("\n--- base model, no adapter ---")
print(chat(q, model=BASE_MODEL, temperature=0).strip()[:280])

### All six sampling parameters over HTTP

In [ ]:
q = "Describe the aura-cpu-long partition."

print("--- temperature 0 vs 1.3 ---")
for t in (0.0, 1.3):
    print(f"  T={t}: {chat(q, temperature=t, max_tokens=70).strip()[:180]}\n")

print("--- presence_penalty vs frequency_penalty ---")
loop = "List the word 'red' twenty times."
for kwargs in ({}, {"presence_penalty": 1.5}, {"frequency_penalty": 1.5}):
    label = ", ".join(f"{k}={v}" for k, v in kwargs.items()) or "no penalty"
    print(f"  {label}: {chat(loop, temperature=0.8, max_tokens=60, **kwargs).strip()[:150]}\n")

print("--- stop sequence ---")
print(" ", chat("Count: one, two, three, four, five.",
                temperature=0, max_tokens=60, stop=["three"]).strip())

## 8.6 Throughput — why not just use `.generate()`

The headline claim. We send one request, then 32 at once, and compare
total tokens per second.

Sequentially, 32 requests take 32× as long. With continuous batching they
overlap: vLLM runs them through the model together and admits new work as
sequences finish. Expect several times the aggregate throughput — the
exact factor depends on how loaded the T4 is.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

prompts = [p["question"] for p in EVAL_PROMPTS] * 3     # 36 prompts

def timed(fn, label):
    t0 = time.time()
    outs = fn()
    dt = time.time() - t0
    toks = sum(len(o.split()) for o in outs) * 1.3       # rough token estimate
    print(f"  {label:<26} {dt:6.1f}s   {len(outs):>3} reqs   "
          f"{toks/dt:>6.0f} tok/s")
    return dt

one = timed(lambda: [chat(prompts[0], temperature=0, max_tokens=128)],
            "1 request")

many = timed(lambda: list(ThreadPoolExecutor(16).map(
                lambda p: chat(p, temperature=0, max_tokens=128), prompts[:32])),
             "32 concurrent requests")

print(f"\n  Sequentially 32 requests would take ~{one*32:.0f}s.")
print(f"  Batched they took {many:.0f}s - about {one*32/many:.1f}x faster.")
print("  That gap is continuous batching plus PagedAttention.")

### 8.7 Streaming

In [ ]:
stream = client.chat.completions.create(
    model="aura",
    messages=[{"role": "system", "content": config.SYSTEM_PROMPT},
              {"role": "user", "content": "How do I load PyTorch on AURA?"}],
    max_tokens=160, temperature=0, stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print("\n\n(tokens arrive as generated - the latency users actually feel)")

## 8.8 Ollama or vLLM?

Not a competition — different jobs.

| | **Ollama** | **vLLM** |
|---|---|---|
| Built for | one user, local | many users, server |
| Format | GGUF | safetensors / GPTQ / AWQ |
| CPU inference | yes | no, GPU only |
| Concurrency | limited | continuous batching |
| Multi-adapter | one at a time | many, per request |
| Setup | one command | flags and tuning |
| Quantization | 2–8 bit K-quants | fp16/bf16, GPTQ, AWQ, fp8 |
| Use when | laptop, demo, prototype, offline | production API, throughput, cost per token |

Rule of thumb: **Ollama to try it, vLLM to ship it.** Many teams use both —
Ollama on developer laptops, vLLM in the cluster.

In [ ]:
server.terminate()
server.wait(timeout=30)
print("vLLM stopped, VRAM released")

## End of the practical

In under four hours you have:

1. Measured a model's real memory footprint against the arithmetic from the slides
2. Watched a base model confidently invent facts about a cluster that doesn't exist
3. Built a dataset in four formats and seen why only two of them survive multi-turn
4. Fine-tuned 0.7% of a 3B model on a free GPU
5. Watched it answer questions it had never seen in that wording
6. Merged, quantized and exported it to GGUF
7. Served it two ways, with the full sampling parameter surface on each

**Take home:** the GGUF, the adapter, and this repo. Swap in your own
dataset at Lab 2 and rerun Labs 2–6 — nothing else has to change.